# GPU baseline experiments

Train the autoencoder, CNN and fixed 50/50 ensemble at 224px. For the final six-method comparison, use Group30_GPU_Full15.ipynb.

Upload only mvtec_anomaly_detection.tar.xz to MyDrive/PRS_Group30, select a Colab GPU runtime, and run in order. Source code is cloned from GitHub. The first run uses bottle; edit CATEGORIES for other categories. Thresholds use normal validation images only.


In [ ]:
import datetime
import json
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/PRS_Group30')
DATA_ARCHIVE = BASE / 'mvtec_anomaly_detection.tar.xz'
CATEGORIES = ['bottle']  # All categories: bottle, cable, capsule, carpet, grid, hazelnut, leather, metal_nut, pill, screw, tile, toothbrush, transistor, wood, zipper
EPOCHS = 10
IMAGE_SIZE = 224
BATCH_SIZE = 16
SEED = 42
assert DATA_ARCHIVE.is_file(), f'Upload dataset: {DATA_ARCHIVE}'


## Install the project and check GPU availability

In [ ]:
PROJECT = Path('/content/Visual_Defect_Detection')
# Keep this checkout unchanged when resuming an experiment.
if not (PROJECT / '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/AndyYeom/Visual_Defect_Detection.git', str(PROJECT)], check=True)
SOURCE_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT, text=True).strip()
print('Source commit:', SOURCE_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(PROJECT) + '[demo,dev]'], check=True)
import torch

print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), 'Select a GPU runtime and run this cell again.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=PROJECT, check=True)


## Extract the dataset
Initial extraction may take a while. Training reads images from the Colab local disk rather than Google Drive.

In [ ]:
LOCAL_ARCHIVE = Path('/content/mvtec_anomaly_detection.tar.xz')
DATA_DIR = Path('/content/mvtec_data')
MARKER = DATA_DIR / '.extraction_complete'
if not MARKER.exists():
    DATA_DIR.mkdir(exist_ok=True)
    shutil.copy2(DATA_ARCHIVE, LOCAL_ARCHIVE)
    with tarfile.open(LOCAL_ARCHIVE) as archive:
        archive.extractall(DATA_DIR, filter='data')
    MARKER.touch()
roots = [DATA_DIR] + [p for p in DATA_DIR.iterdir() if p.is_dir()]
matches = [p for p in roots if (p / 'bottle/train/good').is_dir()]
assert len(matches) == 1, 'Cannot identify MVTec AD dataset root'
DATA_ROOT = matches[0]
for category in CATEGORIES:
    assert (DATA_ROOT / category / 'train/good').is_dir(), category
    assert (DATA_ROOT / category / 'test').is_dir(), category
print('Dataset:', DATA_ROOT)


## Train on normal images and run final evaluation
Each run is saved in a unique directory. Completed models and split manifests are copied to Drive before evaluation. If training is interrupted, restart training for the affected category.

In [ ]:
RUN_ID = datetime.datetime.now(datetime.UTC).strftime('%Y%m%d_%H%M%S_%f')
RUN_LOCAL = Path('/content/vdd_runs') / RUN_ID
RUN_DRIVE = BASE / 'results' / RUN_ID
RUN_LOCAL.mkdir(parents=True)
RUN_DRIVE.mkdir(parents=True)

def run(*args):
    subprocess.run([sys.executable, '-m', 'vdd.cli', *map(str, args)], cwd=PROJECT, check=True)

settings = {'categories': CATEGORIES, 'epochs': EPOCHS, 'size': IMAGE_SIZE,
                'batch_size': BATCH_SIZE, 'seed': SEED, 'weight': 0.5, 'quantile': 0.95,
                'device': torch.cuda.get_device_name(0)}
(RUN_DRIVE / 'settings.json').write_text(json.dumps(settings, indent=2))
(RUN_DRIVE / 'environment.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True))
(RUN_DRIVE / 'source_commit.txt').write_text(SOURCE_COMMIT + '\n')

for category in CATEGORIES:
    folder = RUN_LOCAL / category
    manifest = folder / 'split.json'
    model = folder / 'model'
    report = folder / 'report'
    run('prepare', '--root', DATA_ROOT, '--category', category,
        '--output', manifest, '--seed', SEED, '--validation-fraction', 0.2)
    run('train', '--root', DATA_ROOT, '--manifest', manifest,
        '--output', model, '--epochs', EPOCHS, '--size', IMAGE_SIZE,
        '--batch-size', BATCH_SIZE, '--seed', SEED,
        '--weight', 0.5, '--quantile', 0.95, '--device', 'cuda')
    shutil.copytree(folder, RUN_DRIVE / category, dirs_exist_ok=True)
    run('evaluate', '--root', DATA_ROOT, '--manifest', manifest,
        '--checkpoint', model / 'model.pt', '--output', report, '--device', 'cuda')
    shutil.copytree(folder, RUN_DRIVE / category, dirs_exist_ok=True)
    print(category, json.loads((report / 'metrics.json').read_text())['metrics'])
print('Saved results:', RUN_DRIVE)


## Download results
Download the ZIP below to review performance or use the outputs in subsequent model comparisons. The dataset is not included.

In [ ]:
from google.colab import files

result_zip = shutil.make_archive(str(Path('/content') / ('vdd_results_' + RUN_ID)), 'zip', RUN_DRIVE)
files.download(result_zip)
